# Gratis Fiyat Analizi — Veri Yükleme ve Temizlik

**Proje:** Gratis'te Fiyat Davranışı Analizi: Tüketici için Veri Odaklı Alışveriş Rehberi

Bu notebook'ta:
1. SQLite veritabanından ham veriyi okuyoruz
2. Veri tiplerini düzeltiyoruz (tarih, sayısal alanlar)
3. Mükerrer ve hatalı kayıtları temizliyoruz
4. Panel veri yapısını kuruyoruz (ürün × zaman)
5. Sonraki adımlar için temiz veriyi kaydediyoruz

In [12]:
# Veri işleme
import pandas as pd
import numpy as np

# Veritabanı bağlantısı
import sqlite3

# Görselleştirme
import matplotlib.pyplot as plt
import seaborn as sns

# Yardımcılar
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Görselleştirme ayarları
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print("Kütüphaneler yüklendi ✓")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

Kütüphaneler yüklendi ✓
pandas: 3.0.2
numpy: 2.4.4


In [13]:
# Veritabanı yolunu tanımla
# Notebook notebooks/ klasöründe olduğu için bir üst dizine çıkıp data/ klasörüne giriyoruz
DB_PATH = "../data/gratis.db"

# Bağlantıyı aç ve tüm veriyi pandas DataFrame'e oku
conn = sqlite3.connect(DB_PATH)
df_raw = pd.read_sql_query("SELECT * FROM fiyat_gecmisi", conn)
conn.close()

# Genel görünüm
print(f"Toplam kayıt sayısı: {len(df_raw):,}")
print(f"Sütun sayısı: {df_raw.shape[1]}")
print(f"\nSütunlar:\n{df_raw.columns.tolist()}")
print(f"\nVeri tipleri:")
print(df_raw.dtypes)

Toplam kayıt sayısı: 168,356
Sütun sayısı: 14

Sütunlar:
['id', 'urun_id', 'isim', 'marka', 'kategori', 'fiyat', 'eski_fiyat', 'indirim_yuzde', 'kampanya', 'yorum_sayisi', 'begeni', 'url', 'tarih', 'kayit_zamani']

Veri tipleri:
id                 int64
urun_id              str
isim                 str
marka                str
kategori             str
fiyat            float64
eski_fiyat       float64
indirim_yuzde    float64
kampanya             str
yorum_sayisi       int64
begeni               str
url                  str
tarih                str
kayit_zamani         str
dtype: object


In [14]:
# İlk 5 satıra bak
print("=== İLK 5 SATIR ===")
df_raw.head()

=== İLK 5 SATIR ===


,id,urun_id,isim,marka,kategori,fiyat,eski_fiyat,indirim_yuzde,kampanya,yorum_sayisi,begeni,url,tarih,kayit_zamani
0,1,10209726,Love Generation Lipstick Balm Wet Dream 07 Dar...,Love,Makyaj,229.0,578.0,60.4,250 TL ve Üzeri Alışverişe,7,2B,https://www.gratis.com/ruj/love-generation-lip...,2026-04-04 18:55,2026-04-04 19:28:27.003870
1,2,10215840,Loreal Paris Telescopic Extensionist Maskara,Loreal,Makyaj,679.5,1699.0,60.0,250 TL ve Üzeri Alışverişe,20,8B,https://www.gratis.com/maskara/loreal-paris-te...,2026-04-04 18:55,2026-04-04 19:28:27.003870
2,3,10201916,Maybelline New York Super Lock Brow Glue Kaş S...,Maybelline,Makyaj,520.0,1300.0,60.0,250 TL ve Üzeri Alışverişe,282,58B,https://www.gratis.com/kas-maskarasi/maybellin...,2026-04-04 18:55,2026-04-04 19:28:27.003870
3,4,10209065,Flormar Puffy Liquid Blush Likit Allık 002 Pea...,Flormar,Makyaj,280.0,700.0,60.0,250 TL ve Üzeri Alışverişe,51,13B,https://www.gratis.com/allik/flormar-puffy-liq...,2026-04-04 18:55,2026-04-04 19:28:27.003870
4,5,10212949,Flormar Volume Up Hacim ve Lifting Etkili Yüks...,Flormar,Makyaj,500.0,1250.0,60.0,250 TL ve Üzeri Alışverişe,142,23B,https://www.gratis.com/maskara/flormar-volume-...,2026-04-04 18:55,2026-04-04 19:28:27.003870


In [15]:
# Genel istatistikler ve eksik değer kontrolü
print("=== EKSİK DEĞER (NULL) SAYISI ===")
print(df_raw.isnull().sum())
print(f"\n=== BENZERSİZ ÜRÜN SAYISI: {df_raw['urun_id'].nunique():,} ===")
print(f"=== BENZERSİZ TARİH SAYISI: {df_raw['tarih'].nunique()} ===")
print(f"=== BENZERSİZ KATEGORİ SAYISI: {df_raw['kategori'].nunique()} ===")
print(f"\n=== TARİH ARALIĞI ===")
print(f"En eski: {df_raw['tarih'].min()}")
print(f"En yeni: {df_raw['tarih'].max()}")

=== EKSİK DEĞER (NULL) SAYISI ===
id                    0
urun_id               0
isim                  0
marka                 0
kategori              0
fiyat                 0
eski_fiyat        17494
indirim_yuzde     17494
kampanya         129321
yorum_sayisi          0
begeni              201
url                   0
tarih                 0
kayit_zamani          0
dtype: int64

=== BENZERSİZ ÜRÜN SAYISI: 11,679 ===
=== BENZERSİZ TARİH SAYISI: 686 ===
=== BENZERSİZ KATEGORİ SAYISI: 14 ===

=== TARİH ARALIĞI ===
En eski: 2026-04-04 18:55
En yeni: 2026-05-06 20:27


In [16]:
# Mükerrer kayıt kontrolü
# Aynı (urun_id, tarih) çifti birden fazla kez var mı?
duplicate_count = df_raw.duplicated(subset=["urun_id", "tarih"]).sum()
print(f"Mükerrer (urun_id + tarih) sayısı: {duplicate_count:,}")
print(f"Toplam kayıt: {len(df_raw):,}")
print(f"Mükerrer oranı: %{duplicate_count / len(df_raw) * 100:.2f}")

# Eğer mükerrer varsa, kaç farklı urun_id etkilenmiş?
if duplicate_count > 0:
    affected_products = df_raw[df_raw.duplicated(subset=["urun_id", "tarih"], keep=False)]["urun_id"].nunique()
    print(f"Etkilenen benzersiz ürün sayısı: {affected_products:,}")
    print("\nÖrnek mükerrer kayıtlar:")
    print(df_raw[df_raw.duplicated(subset=["urun_id", "tarih"], keep=False)].head(6)[["urun_id", "tarih", "fiyat", "eski_fiyat", "kayit_zamani"]])

Mükerrer (urun_id + tarih) sayısı: 0
Toplam kayıt: 168,356
Mükerrer oranı: %0.00


In [17]:
# Çalışma kopyası oluştur
df = df_raw.copy()

# === 1) TARİH SÜTUNLARINI DATETIME'A ÇEVİR ===
df["tarih"] = pd.to_datetime(df["tarih"], format="%Y-%m-%d %H:%M", errors="coerce")
df["kayit_zamani"] = pd.to_datetime(df["kayit_zamani"], errors="coerce")

# Gün bazında yeni bir sütun oluştur (saat/dakika bilgisini at)
df["snapshot_gun"] = df["tarih"].dt.normalize()  # 2026-04-04 18:55 → 2026-04-04 00:00

# === 2) BEĞENI SÜTUNUNU SAYISALA ÇEVİR ===
# "383" → 383, "2B" → 2000, "13B" → 13000, "1.5B" gibi durumlar için de hazırlık
def begeni_parse(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().upper()
    if s.endswith("B"):  # Bin
        try:
            return float(s[:-1].replace(",", ".")) * 1000
        except:
            return np.nan
    if s.endswith("M"):  # Milyon (olur mu bilmem ama hazır olalım)
        try:
            return float(s[:-1].replace(",", ".")) * 1_000_000
        except:
            return np.nan
    try:
        return float(s.replace(",", "."))
    except:
        return np.nan

df["begeni_sayi"] = df["begeni"].apply(begeni_parse)

# === 3) İNDİRİM YOK = 0 olarak doldur ===
# eski_fiyat NULL ise ürün indirimde değil, normal fiyatından satılıyor
# Bu yüzden indirim_yuzde de 0 olmalı, eski_fiyat da o anki fiyata eşit
df["indirim_yuzde"] = df["indirim_yuzde"].fillna(0)
df["eski_fiyat"] = df["eski_fiyat"].fillna(df["fiyat"])

# === 4) İNDİRİMDE Mİ? boolean flag ===
df["indirimde_mi"] = df["indirim_yuzde"] > 0

# === 5) KAMPANYA NULL → "Kampanya Yok" ===
df["kampanya"] = df["kampanya"].fillna("Kampanya Yok")

# === 6) GEREKSİZ SÜTUNLARI AT ===
df = df.drop(columns=["id"])  # autoincrement ID, analiz için gereksiz

# === SONUÇ ===
print("=== TEMİZLENMİŞ VERİ ÖZETİ ===")
print(f"Satır sayısı: {len(df):,}")
print(f"Sütun sayısı: {df.shape[1]}")
print(f"\nYeni veri tipleri:")
print(df.dtypes)
print(f"\nEksik değerler:")
print(df.isnull().sum())

=== TEMİZLENMİŞ VERİ ÖZETİ ===
Satır sayısı: 168,356
Sütun sayısı: 16

Yeni veri tipleri:
urun_id                     str
isim                        str
marka                       str
kategori                    str
fiyat                   float64
eski_fiyat              float64
indirim_yuzde           float64
kampanya                    str
yorum_sayisi              int64
begeni                      str
url                         str
tarih            datetime64[us]
kayit_zamani     datetime64[us]
snapshot_gun     datetime64[us]
begeni_sayi             float64
indirimde_mi               bool
dtype: object

Eksik değerler:
urun_id            0
isim               0
marka              0
kategori           0
fiyat              0
eski_fiyat         0
indirim_yuzde      0
kampanya           0
yorum_sayisi       0
begeni           201
url                0
tarih              0
kayit_zamani       0
snapshot_gun       0
begeni_sayi      201
indirimde_mi       0
dtype: int64


In [7]:
# === ÜRÜN × GÜN BAZINA İNDİRGEME ===
# Aynı ürünün aynı gün içindeki birden fazla snapshot'ından
# kayit_zamani en yeni olanı tutuyoruz (o günün en güncel fiyatı)

print("İndirgeme öncesi satır sayısı:", f"{len(df):,}")

# Önce kayit_zamani'ya göre azalan sırala, sonra duplikatları at (en yenisi kalır)
df_daily = (
    df.sort_values("kayit_zamani", ascending=False)
      .drop_duplicates(subset=["urun_id", "snapshot_gun"], keep="first")
      .sort_values(["urun_id", "snapshot_gun"])
      .reset_index(drop=True)
)

print("İndirgeme sonrası satır sayısı:", f"{len(df_daily):,}")
print(f"Çıkarılan satır: {len(df) - len(df_daily):,}")
print(f"\nBenzersiz ürün sayısı: {df_daily['urun_id'].nunique():,}")
print(f"Benzersiz gün sayısı: {df_daily['snapshot_gun'].nunique()}")

# Her ürünün kaç gün gözlemlendiğine bak
gozlem_sayisi = df_daily.groupby("urun_id").size()
print(f"\n=== ÜRÜN BAŞINA GÖZLEM SAYISI DAĞILIMI ===")
print(gozlem_sayisi.describe())
print(f"\nSadece 1 gün gözlemlenen ürün sayısı: {(gozlem_sayisi == 1).sum():,}")
print(f"5+ gün gözlemlenen ürün sayısı: {(gozlem_sayisi >= 5).sum():,}")
print(f"10+ gün gözlemlenen ürün sayısı: {(gozlem_sayisi >= 10).sum():,}")

İndirgeme öncesi satır sayısı: 168,356
İndirgeme sonrası satır sayısı: 121,282
Çıkarılan satır: 47,074

Benzersiz ürün sayısı: 11,679
Benzersiz gün sayısı: 15

=== ÜRÜN BAŞINA GÖZLEM SAYISI DAĞILIMI ===
count    11679.000000
mean        10.384622
std          3.372494
min          1.000000
25%          9.000000
50%         11.000000
75%         13.000000
max         15.000000
dtype: float64

Sadece 1 gün gözlemlenen ürün sayısı: 351
5+ gün gözlemlenen ürün sayısı: 10,628
10+ gün gözlemlenen ürün sayısı: 8,396


In [18]:
# === YETERLİ GÖZLEMİ OLAN ÜRÜNLERİ FİLTRELE ===
# ML modelleri için ürün başına en az 5 gün gözlem gerekli

MIN_GOZLEM = 5

# Hangi ürünlerin yeterli gözlemi var?
yeterli_urunler = gozlem_sayisi[gozlem_sayisi >= MIN_GOZLEM].index

print(f"Filtreleme öncesi ürün sayısı: {df_daily['urun_id'].nunique():,}")
print(f"Filtreleme sonrası ürün sayısı: {len(yeterli_urunler):,}")
print(f"Atılan ürün sayısı: {df_daily['urun_id'].nunique() - len(yeterli_urunler):,}")

# Filtreyi uygula
df_clean = df_daily[df_daily["urun_id"].isin(yeterli_urunler)].copy().reset_index(drop=True)

print(f"\nFiltreleme öncesi satır sayısı: {len(df_daily):,}")
print(f"Filtreleme sonrası satır sayısı: {len(df_clean):,}")

# Son durumu özetle
print(f"\n=== FİNAL TEMİZ VERİ ===")
print(f"Satır sayısı: {len(df_clean):,}")
print(f"Ürün sayısı: {df_clean['urun_id'].nunique():,}")
print(f"Kategori sayısı: {df_clean['kategori'].nunique()}")
print(f"Marka sayısı: {df_clean['marka'].nunique():,}")
print(f"Gün sayısı: {df_clean['snapshot_gun'].nunique()}")
print(f"Tarih aralığı: {df_clean['snapshot_gun'].min().date()} → {df_clean['snapshot_gun'].max().date()}")

# Hızlı sanity check: ürün başına gözlem dağılımı
print(f"\nÜrün başına ortalama gözlem: {df_clean.groupby('urun_id').size().mean():.1f} gün")

Filtreleme öncesi ürün sayısı: 11,679
Filtreleme sonrası ürün sayısı: 10,628
Atılan ürün sayısı: 1,051

Filtreleme öncesi satır sayısı: 121,282
Filtreleme sonrası satır sayısı: 118,812

=== FİNAL TEMİZ VERİ ===
Satır sayısı: 118,812
Ürün sayısı: 10,628
Kategori sayısı: 14
Marka sayısı: 422
Gün sayısı: 15
Tarih aralığı: 2026-04-04 → 2026-05-06

Ürün başına ortalama gözlem: 11.2 gün


In [19]:
# Klasör oluştur ve veriyi kaydet
import os

# processed klasörü yoksa oluştur
os.makedirs("../data/processed", exist_ok=True)

# Parquet olarak kaydet
output_path = "../data/processed/gratis_clean.parquet"

try:
    df_clean.to_parquet(output_path, index=False, engine="pyarrow")
    print(f"✓ Parquet olarak kaydedildi: {output_path}")
except ImportError:
    # pyarrow yoksa, fastparquet veya csv'ye fallback
    print("pyarrow kurulu değil, CSV olarak kaydediyorum...")
    output_path = "../data/processed/gratis_clean.csv"
    df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"✓ CSV olarak kaydedildi: {output_path}")

# Dosya boyutunu yaz
size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Dosya boyutu: {size_mb:.2f} MB")

# Doğrulama: tekrar oku ve karşılaştır
if output_path.endswith(".parquet"):
    df_test = pd.read_parquet(output_path)
else:
    df_test = pd.read_csv(output_path)

print(f"\nDoğrulama: Okunan satır sayısı = {len(df_test):,} (orijinal: {len(df_clean):,})")
print(f"Sütunlar eşleşiyor: {list(df_test.columns) == list(df_clean.columns)}")

✓ Parquet olarak kaydedildi: ../data/processed/gratis_clean.parquet
Dosya boyutu: 1.61 MB

Doğrulama: Okunan satır sayısı = 118,812 (orijinal: 118,812)
Sütunlar eşleşiyor: True


In [20]:
import os

# processed klasörü yoksa oluştur
os.makedirs("../data/processed", exist_ok=True)

# CSV olarak kaydet (pyarrow'a girmiyoruz)
output_path = "../data/processed/gratis_clean.csv"
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

# Dosya boyutu ve doğrulama
size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"✓ CSV olarak kaydedildi: {output_path}")
print(f"Dosya boyutu: {size_mb:.2f} MB")

# Doğrulama: tekrar oku ve karşılaştır
df_test = pd.read_csv(output_path, parse_dates=["tarih", "kayit_zamani", "snapshot_gun"])
print(f"\nDoğrulama:")
print(f"  Okunan satır: {len(df_test):,} (orijinal: {len(df_clean):,})")
print(f"  Sütunlar eşleşiyor: {list(df_test.columns) == list(df_clean.columns)}")
print(f"  Tarih tipi: {df_test['snapshot_gun'].dtype}")

✓ CSV olarak kaydedildi: ../data/processed/gratis_clean.csv
Dosya boyutu: 33.15 MB

Doğrulama:
  Okunan satır: 118,812 (orijinal: 118,812)
  Sütunlar eşleşiyor: True
  Tarih tipi: datetime64[us]


## Özet — Veri Yükleme ve Temizlik

**Başlangıç:** SQLite veritabanında 168,356 ham kayıt  
**Sonuç:** 118,812 temiz gözlem, 10,628 ürün, 14 kategori, 422 marka, 15 snapshot günü

### Yapılan İşlemler
1. **Veri kaynağı:** Scraper tarafından oluşturulan `gratis.db` SQLite veritabanından okuma
2. **Veri tipi dönüşümleri:** Tarihler `datetime`'a, `begeni` ("2B", "13B") sayısala çevrildi
3. **Eksik değer stratejisi:**
   - `eski_fiyat` ve `indirim_yuzde` boşları → "indirim yok" anlamına geliyor, 0/güncel fiyatla dolduruldu
   - `kampanya` boşları → "Kampanya Yok" etiketi atandı
4. **Türetilmiş sütunlar:** `snapshot_gun` (gün hassasiyetinde tarih), `indirimde_mi` (boolean flag), `begeni_sayi` (sayısal beğeni)
5. **Gün bazına indirgeme:** Aynı ürün-aynı günün birden fazla snapshot'ı varsa en güncel olan tutuldu (47K satır azaltıldı)
6. **Minimum gözlem filtresi:** En az 5 gün gözlemlenmiş ürünler tutuldu — ML modelleri için güvenilir özellik çıkarımı amacıyla (%9 ürün kaybı kabul edildi)

### Veri Kalitesi Notları
- Mükerrer (`urun_id` + `tarih`) kayıt: **0** ✓
- Ürün başına ortalama gözlem: **11.2 gün**
- 8,396 ürün ≥10 gün gözlemlendi → ML için sağlam temel

### Sonraki Adım
`02_kesifsel_analiz.ipynb` — Keşifsel Veri Analizi (EDA)